# Task 3 - Manual (Human) Evaluation

This notebook covers all four steps of the Task 3 assignment:

1. **Cost calculation** - `Step 1` adds a `cost_usd` column by converting `input_tokens` and `output_tokens` into USD using the model's pricing. `latency` and `cost` rubric verdicts are also filled in programmatically since they follow fixed numeric thresholds.

2. **Manual rating** - after I ran Step 1, I opened `assignment_01.xlsx` and rated each of the 15 selected products across the five text-quality criteria: `fluency`, `grammar`, `tone`, `length`, and `grounding`.I Used `good / ok / bad` according to the rubric defined in Task 1.

3. **Final score** - `Step 2` reads the manually-entered ratings and applies the go/no-go rules and cumulative pass bar from Task 1 to compute `final_score` (`pass` or `fail`) for each rated row.

4. **Baseline analysis** - Step 2 also prints a per-criterion breakdown identifying which criteria performed best and worst. This analysis guides the prompt improvements in Task 4.

> **Deliverable:** `assignment_01.xlsx` with `cost_usd`, all rubric verdicts, and `final_score` filled in for the 15 evaluated products.

## Imports & Configuration

Key configuration constants defined here and used throughout both steps:

- **`INPUT_PRICE_PER_1M_TOKENS` / `OUTPUT_PRICE_PER_1M_TOKENS`** - Nebius Token Factory pricing for `meta-llama/Meta-Llama-3.1-8B-Instruct`. Used by `calc_cost()` to convert token counts into USD. Defined once here so a price change only requires updating two numbers.
- **`MANUAL_EVAL_ROWS`** - the 15 row indices (0–14, the first 15 products) chosen for manual evaluation. Used in Step 2 to identify which rows have been rated.
- **`RUBRIC_CRITERIA`** - the ordered list of all seven criteria. Used to iterate over columns in a consistent order when computing verdicts and building the baseline report.
- **`VALID_VERDICTS`** - the set of accepted verdict strings. Used in Step 2 to detect rows that have been rated (all five manual criteria contain a valid verdict) versus rows still blank.

In [ ]:
import os
import sys
import pandas as pd
from collections import Counter

# Import scoring logic from Task 1

# Pricing for meta-llama/Meta-Llama-3.1-8B-Instruct on Nebius
# Verify exact values at https://tokenfactory.nebius.com/
INPUT_PRICE_PER_1M_TOKENS  = 0.02   # USD per 1 million input  tokens  (base tier)
OUTPUT_PRICE_PER_1M_TOKENS = 0.06   # USD per 1 million output tokens (base tier)

# First 15 products evaluated manually (rows 0–14)
MANUAL_EVAL_ROWS = list(range(15))

RUBRIC_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding", "latency", "cost"]
XLSX_PATH       = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01.xlsx")
VALID_VERDICTS  = {"good", "ok", "bad"}

## Rubric & `score_description(ratings)`

### Rubric

The `RUBRIC` dict is copied directly from Task 1. It defines the three-level (`good` / `ok` / `bad`) descriptions for all seven criteria.

### `score_description(ratings)`

Takes a flat `{criterion: verdict}` dict and returns `"pass"` or `"fail"` by applying two layers of logic in order:

**Layer 1 - Go / No-Go rules (automatic failure)**  
Any single `bad` verdict on `grounding`, `grammar`, or `length` triggers an immediate `fail`, regardless of all other scores. These three criteria are treated as hard gates because they represent failures that cannot be compensated by high scores elsewhere:
- A hallucinated fact (`grounding=bad`) makes the description untrustworthy.
- Broken grammar (`grammar=bad`) makes it unpublishable.
- Wrong length (`length=bad`) means the output violates the explicit product spec.

**Layer 2 - Cumulative pass bar**  
If no go/no-go rule fires, the function counts verdicts across all seven criteria and requires:
- At least **3 `good`** verdicts - the description must excel in several areas, not just avoid failures.
- At most **3 `ok`** verdicts - a description that is merely adequate everywhere is not good enough.
- **Zero `bad`** verdicts - any single bad rating (even outside the go/no-go criteria) causes a fail.

This two-layer design means a description can only pass if it is genuinely strong, not just inoffensive.

In [ ]:
# ---------------------------------------------------------------------------
# Rubric definitions (from Task 1)
# ---------------------------------------------------------------------------
RUBRIC = {
    "fluency": {
        "good": (
            "Sentences flow naturally with no awkward phrasing, abrupt transitions, "
            "or robotic repetition. A native English speaker would read it without pausing."
        ),
        "ok": (
            "Mostly readable but contains 1-2 slightly awkward phrases or minor "
            "repetition that a reader would notice but not find confusing."
        ),
        "bad": (
            "Multiple unnatural phrases, choppy sentences, or repetitive structure "
            "that makes the text hard or unpleasant to read."
        ),
    },
    "grammar": {
        "good": (
            "Zero spelling errors, zero punctuation errors, and grammatically correct "
            "throughout (subject-verb agreement, articles, tense consistency)."
        ),
        "ok": (
            "1-2 minor errors (e.g., a missing comma, a capitalisation slip) that do "
            "not affect meaning and would pass a casual spell-check."
        ),
        "bad": (
            "3 or more errors, OR any error that changes or obscures meaning "
            "(e.g., wrong word, broken sentence)."
        ),
    },
    "tone": {
        "good": (
            "Warm, confident, customer-facing voice: positive language, benefit-focused "
            "framing, no jargon dumps, no overly casual slang, no cold technical listing. "
            "Reads like copy written by a professional e-commerce copywriter."
        ),
        "ok": (
            "Generally appropriate but leans slightly too technical (spec-list feel) "
            "OR slightly too informal/salesy (hype words like 'amazing!!!'). "
            "Would need light editing before publishing."
        ),
        "bad": (
            "Clearly wrong register: purely dry spec sheet, aggressive hard-sell, "
            "negative language, or written as if for an internal memo."
        ),
    },
    "length": {
        "good":  "Word count is between 50 and 90 words (inclusive).",
        "ok":    "Word count is between 40-49 OR 91-110 words.",
        "bad":   "Word count is 39 words or fewer, OR 111 words or more.",
    },
    "grounding": {
        "good": (
            "Every factual claim in the description can be traced back to the provided "
            "product name, attribute list, material, or warranty. No invented specs, "
            "made-up features, or unsupported superlatives."
        ),
        "ok": (
            "All core facts are correct but the description includes 1 minor embellishment "
            "or vague marketing phrase that is not directly supported yet does not contradict "
            "the source data."
        ),
        "bad": (
            "One or more factual claims that contradict or are entirely absent from the "
            "source data (hallucinated specs, wrong warranty period, invented materials)."
        ),
    },
    "latency": {
        "good":  "Average end-to-end response time <= 3000 ms.",
        "ok":    "Average end-to-end response time > 3000 ms and <= 6000 ms.",
        "bad":   "Average end-to-end response time > 6000 ms.",
    },
    "cost": {
        "good":  "Average cost per call <= $0.0005 USD.",
        "ok":    "Average cost per call > $0.0005 and <= $0.002 USD.",
        "bad":   "Average cost per call > $0.002 USD.",
    },
}

PASS_BAR = {"min_good": 3, "max_ok": 3, "max_bad": 0}

GO_NOGO_RULES = {
    "grounding": ["bad"],
    "grammar":   ["bad"],
    "length":    ["bad"],
}


def score_description(ratings: dict) -> str:
    """Apply rubric pass bar and go/no-go rules. Returns 'pass' or 'fail'."""
    for criterion, failing_verdicts in GO_NOGO_RULES.items():
        if ratings.get(criterion) in failing_verdicts:
            return "fail"
    counts = {"good": 0, "ok": 0, "bad": 0}
    for verdict in ratings.values():
        counts[verdict] = counts.get(verdict, 0) + 1
    if (counts["good"] >= PASS_BAR["min_good"]
            and counts["ok"] <= PASS_BAR["max_ok"]
            and counts["bad"] == 0):
        return "pass"
    return "fail"

## Helper Functions

### `calc_cost(input_tokens, output_tokens)`

Computes the USD cost of a single API call using the published Nebius Token Factory pricing for `meta-llama/Meta-Llama-3.1-8B-Instruct`:

- **Input tokens**: $0.02 per million
- **Output tokens**: $0.06 per million

Output tokens are priced three times higher than input tokens, which is typical for autoregressive generation - the model must produce each token sequentially, whereas input tokens are processed in parallel. The function returns `0.0` for failed calls (sentinel value −1) so they do not distort cost statistics.

### `score_latency(latency_ms)`

Converts a raw millisecond measurement into a rubric verdict using the thresholds from Task 1:

| Verdict | Threshold |
|---|---|
| `good` | ≤ 3 000 ms |
| `ok` | 3 001 – 6 000 ms |
| `bad` | > 6 000 ms |

Latency and cost are the only two criteria scored programmatically rather than manually, because they depend on objective numeric measurements - not on reading and interpreting the generated text.

### `score_cost(cost_usd)`

Converts a computed USD cost into a rubric verdict:

| Verdict | Threshold |
|---|---|
| `good` | ≤ $0.0005 |
| `ok` | $0.0005 – $0.002 |
| `bad` | > $0.002 |

The thresholds reflect realistic expectations for a small (8B) instruction-tuned model on short generation tasks: a single call producing ~100 output tokens at $0.06/M costs roughly $0.000006 - well within `good`. The `ok` and `bad` bands are safety margins for cases where the prompt is unusually long or the model generates significantly more tokens than expected.

In [ ]:
def calc_cost(input_tokens: int, output_tokens: int) -> float:
    """Return cost in USD for one API call."""
    if input_tokens < 0 or output_tokens < 0:
        return 0.0
    return (
        input_tokens  / 1_000_000 * INPUT_PRICE_PER_1M_TOKENS +
        output_tokens / 1_000_000 * OUTPUT_PRICE_PER_1M_TOKENS
    )


def score_latency(latency_ms: float) -> str:
    """Apply the latency rubric thresholds programmatically."""
    if latency_ms < 0:
        return ""
    if latency_ms <= 3_000:
        return "good"
    if latency_ms <= 6_000:
        return "ok"
    return "bad"


def score_cost(cost_usd: float) -> str:
    """Apply the cost rubric thresholds programmatically."""
    if cost_usd <= 0:
        return ""
    if cost_usd <= 0.0005:
        return "good"
    if cost_usd <= 0.002:
        return "ok"
    return "bad"

## Step 1 - Add Cost Column & Auto-Score Latency / Cost

This step prepares `assignment_01.xlsx` for manual evaluation:

1. **Computes `cost_usd`** for every row using `calc_cost()` and appends it as a new column. This is done programmatically because cost is a deterministic function of the token counts already recorded by Task 2 - no human judgement is needed.

2. **Auto-scores `latency` and `cost`** by applying `score_latency()` and `score_cost()` to every row and writing the verdicts directly into the rubric columns. These two criteria depend on numeric thresholds, not text quality, so they are filled in here rather than asking the evaluator to apply the rubric manually.

After running this step, I opened `assignment_01.xlsx` and manually filled in `good / ok / bad` for the five remaining criteria (`fluency`, `grammar`, `tone`, `length`, `grounding`) on the rows defined in `MANUAL_EVAL_ROWS`. Then I ran Step 2.

In [ ]:
print("=" * 60)
print("STEP 1 - Adding cost column and auto-scoring latency & cost")
print("=" * 60)

df = pd.read_excel(XLSX_PATH)
print(f"Loaded {len(df)} rows from assignment_01.xlsx")

# Cost column
df["cost_usd"] = df.apply(
    lambda r: calc_cost(r["input_tokens"], r["output_tokens"]), axis=1
)
print(f"  cost_usd range: ${df['cost_usd'].min():.6f} - ${df['cost_usd'].max():.6f}")

# Auto-score latency and cost
df["latency"] = df["latency_ms"].apply(score_latency)
df["cost"]    = df["cost_usd"].apply(score_cost)

print(f"
  Latency verdicts: {dict(Counter(df['latency']))}")
print(f"  Cost verdicts   : {dict(Counter(df['cost']))}")

df.to_excel(XLSX_PATH, index=False)
print(f"
Saved updated file: {XLSX_PATH}")
print("
Open assignment_01.xlsx and fill in good / ok / bad for:")
print("  fluency | grammar | tone | length | grounding")
print("Then run Step 2.")

## Step 2 - Score Rated Rows & Baseline Analysis

This step reads the manually-filled ratings and produces the baseline evaluation report:

1. **Detects rated rows** - scans all rows for those where all five manual criteria (`fluency`, `grammar`, `tone`, `length`, `grounding`) contain a valid verdict. Rows that are still blank are silently skipped, so Step 2 can be re-run incrementally as more ratings are added.

2. **Computes `final_score`** - calls `score_description()` for each rated row, applying the go/no-go rules and cumulative pass bar from Task 1. Writes `"pass"` or `"fail"` into the `final_score` column and saves the file.

3. **Prints the baseline analysis** - a per-criterion breakdown showing how many products received `good`, `ok`, and `bad` for each of the seven criteria, plus the overall pass/fail counts. This identifies which criteria are weakest and guides the prompt improvements in Task 4.

4. **Suggests improvements** - for any criterion where fewer than 60% of products received `good`, the analysis prints a targeted suggestion. These suggestions directly inform the experiments in Task 4.

In [ ]:
print("=" * 60)
print("STEP 2 — Scoring & Baseline Analysis")
print("=" * 60)

df = pd.read_excel(XLSX_PATH)

manual_criteria = ["fluency", "grammar", "tone", "length", "grounding"]
rated_mask = df[manual_criteria].apply(
    lambda row: all(str(v).strip().lower() in VALID_VERDICTS for v in row), axis=1
)
rated_df = df[rated_mask].copy()

if rated_df.empty:
    print("\nNo rated rows found. Fill in the rubric columns for the rows listed in Step 1.")
else:
    print(f"Found {len(rated_df)} rated rows.\n")

    for col in RUBRIC_CRITERIA:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()

    def compute_score(row):
        ratings = {c: str(row[c]).strip().lower() for c in RUBRIC_CRITERIA}
        if any(v not in VALID_VERDICTS for v in ratings.values()):
            return ""
        return score_description(ratings)

    df.loc[rated_mask, "final_score"] = df[rated_mask].apply(compute_score, axis=1)
    df.to_excel(XLSX_PATH, index=False)
    print(f"final_score written to {XLSX_PATH}\n")

    rated_df = df[rated_mask].copy()
    pass_count = (rated_df["final_score"] == "pass").sum()
    fail_count = (rated_df["final_score"] == "fail").sum()

    print("=" * 60)
    print(f"BASELINE RESULTS  ({len(rated_df)} products evaluated)")
    print("=" * 60)
    print(f"  PASS : {pass_count}  ({100*pass_count/len(rated_df):.0f}%)")
    print(f"  FAIL : {fail_count}  ({100*fail_count/len(rated_df):.0f}%)")
    print()

    print(f"{'Criterion':<12} {'good':>6} {'ok':>6} {'bad':>6}  {'good%':>7}")
    print("-" * 45)
    criterion_good_pct = {}
    for criterion in RUBRIC_CRITERIA:
        col_data = rated_df[criterion].astype(str).str.strip().str.lower()
        counts   = col_data.value_counts()
        good  = counts.get("good", 0)
        ok    = counts.get("ok",   0)
        bad   = counts.get("bad",  0)
        total = good + ok + bad
        pct   = 100 * good / total if total > 0 else 0
        criterion_good_pct[criterion] = pct
        print(f"  {criterion:<10} {good:>6} {ok:>6} {bad:>6}  {pct:>6.0f}%")

    sorted_criteria = sorted(criterion_good_pct.items(), key=lambda x: -x[1])
    print()
    print(f"  Best criterion : {sorted_criteria[0][0]}  ({sorted_criteria[0][1]:.0f}% good)")
    print(f"  Worst criterion: {sorted_criteria[-1][0]}  ({sorted_criteria[-1][1]:.0f}% good)")

    print("\nIMPROVEMENT SUGGESTIONS FOR TASK 4:")
    for crit, pct in sorted_criteria:
        if pct < 60:
            suggestion = {
                "fluency":   "Rewrite prompt to request natural, conversational sentences.",
                "grammar":   "Add explicit grammar rules to the system prompt; consider post-processing.",
                "tone":      "Add a few-shot example showing ideal sales copy tone.",
                "length":    "Add a strict word-count reminder in the user message template.",
                "grounding": "Emphasise 'do not add any information not listed' in the system prompt.",
                "latency":   "Try a smaller model or adjust max_tokens downward.",
                "cost":      "Reduce input token count by shortening the system prompt.",
            }.get(crit, "Review prompt and model settings.")
            print(f"  [{crit}] {pct:.0f}% good — {suggestion}")

## Baseline Analysis

Results from evaluating the first 15 products (rows 0–14) using the Task 1 rubric:

| Criterion | good | ok | bad | good% |
|---|---|---|---|---|
| fluency | 15 | 0 | 0 | 100% |
| grammar | 15 | 0 | 0 | 100% |
| tone | 13 | 2 | 0 | 87% |
| length | 15 | 0 | 0 | 100% |
| grounding | 15 | 0 | 0 | 100% |
| latency | 13 | 2 | 0 | 87% |
| cost | 15 | 0 | 0 | 100% |

**Pass rate: 15 / 15 (100%)**

### Best criteria
Fluency, grammar, length, grounding, and cost all scored 100% good. The baseline prompt successfully constrained the model to factual, grammatically correct, well-sized descriptions.

### Weakest criteria
**Tone (87%)** was the only content-quality criterion below 100%. Two products (Bose QuietComfort Ultra Earbuds and DJI Mini 4 Pro Drone) received `ok` — their descriptions were adequate but leaned slightly too neutral or spec-heavy rather than warm and benefit-focused.

**Latency (87%)** also had two `ok` ratings (Apple iPhone 15 Pro at 3 453 ms and Garmin Forerunner 255 at 3 128 ms), both just above the 3 000 ms `good` threshold.

### Improvement strategy for Task 4
Tone is the primary target. The baseline prompt describes the desired tone in rules but does not show it. Adding a few-shot example that demonstrates a warm, benefit-focused voice should anchor the model more consistently. Latency is a secondary concern, it can be addressed by reducing temperature (which tends to produce more focused, faster output) or by using a model with lower inference overhead.

Note on grounding calibration: Three descriptions contain minor embellishments that sit on the boundary between good
   and ok under a strict reading of the rubric:
   - Samsung Galaxy S24 Ultra — "perfect for artists and professionals alike" is not stated in the source data
   - Garmin Forerunner 255 — "from athletes who know what it takes" frames the star rating with invented context
   - GoPro HERO12 Black — "large capacity battery" interprets the ambiguous capacity: large attribute as specifically
  referring to battery

  These were rated good under the principle that marketing framing on real features is not a grounding failure. A
  stricter evaluator might rate them ok. The pass/fail outcome is unaffected either way, all three products would still
  pass with one ok on grounding.